# 03 — Modeling

Trains the heuristic baseline, Logistic Regression, Random Forest, and LightGBM on a **time-based split** and compares them on ROC-AUC, PR-AUC, Brier score, and precision@k. See `scripts/run_pipeline.py` for the scripted (non-notebook) equivalent used in CI.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from po_delay.data_generation import GeneratorParams, generate
from po_delay.features import build_features
from po_delay.models import (
    build_logistic_pipeline,
    build_random_forest_pipeline,
    evaluate,
    fit_lightgbm_with_early_stopping,
    heuristic_baseline_scores,
)
from po_delay.validation import assert_no_time_leakage, time_based_split

df = generate(GeneratorParams(n_orders=40_000, n_suppliers=150, seed=42))
X, y = build_features(df)
split = time_based_split(df)
assert_no_time_leakage(df, split)

X_train, y_train = X.iloc[split.train_idx], y.iloc[split.train_idx]
X_val, y_val = X.iloc[split.val_idx], y.iloc[split.val_idx]
X_test, y_test = X.iloc[split.test_idx], y.iloc[split.test_idx]
len(X_train), len(X_val), len(X_test)

(28000, 6000, 6000)

In [2]:
results = {}
results["heuristic_baseline"] = evaluate(y_test, heuristic_baseline_scores(X_test))

logit = build_logistic_pipeline().fit(X_train, y_train)
results["logistic_regression"] = evaluate(y_test, logit.predict_proba(X_test)[:, 1])

rf = build_random_forest_pipeline().fit(X_train, y_train)
results["random_forest"] = evaluate(y_test, rf.predict_proba(X_test)[:, 1])

lgbm = fit_lightgbm_with_early_stopping(X_train, y_train, X_val, y_val)
lgbm_proba = lgbm.predict_proba(X_test)[:, 1]
results["lightgbm"] = evaluate(y_test, lgbm_proba)

pd.DataFrame(results).T.sort_values("pr_auc", ascending=False)

,roc_auc,pr_auc,brier_score,precision_at_5pct,precision_at_10pct,precision_at_20pct,base_rate
lightgbm,0.678982,0.374834,0.227874,0.496667,0.458333,0.387500,0.2145
logistic_regression,0.675580,0.371635,0.222165,0.503333,0.455000,0.391667,0.2145
random_forest,0.673842,0.364918,0.227167,0.493333,0.438333,0.386667,0.2145
heuristic_baseline,0.622860,0.286928,0.176333,0.453333,0.383333,0.355833,0.2145


## Observed result and an honest caveat

LightGBM and Random Forest both beat the heuristic baseline on ranking quality (ROC-AUC, PR-AUC, precision@k). Their Brier scores are *worse* than the heuristic's, though — a side effect of using `class_weight="balanced"` to improve ranking under class imbalance, which shifts predicted probabilities away from being well-calibrated. A production deployment would want an explicit recalibration step (e.g., `sklearn.calibration.CalibratedClassifierCV` with isotonic or Platt scaling) fit on the validation set before trusting the raw probabilities for the risk-tier thresholds in `decision_rules.py`. This is called out as a limitation in `docs/model_card.md` rather than papered over.